In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "320_build_gold_fact_project_item_estimate.py"
RUN_ID = str(uuid.uuid4())

FACT_ESTIMATE_ITEM_TABLE = f"{catalog}.silver.fact_estimate_item"
FACT_ESTIMATE_PROJECT_TABLE = f"{catalog}.silver.fact_estimate_project"
DIM_PROJECT_TABLE = f"{catalog}.silver.dim_project"
DIM_COUNTY_TABLE = f"{catalog}.silver.dim_county"
DIM_ITEM_SPEC_TABLE = f"{catalog}.silver.dim_item_specification"
DIM_ITEM_WORK_CATEGORY_TABLE = f"{catalog}.gold.dim_item_work_category"
TARGET_TABLE = f"{catalog}.gold.fact_project_item_estimate"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.fact_project_item_estimate
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    AS
    WITH work_category_map AS (
        SELECT
              effective_specification_description_standardized
            , item_work_category
            , item_work_category_source
            , is_defaulted_work_category
        FROM (
            SELECT
                  effective_specification_description_standardized
                , item_work_category
                , item_work_category_source
                , is_defaulted_work_category
                , ROW_NUMBER() OVER (
                    PARTITION BY effective_specification_description_standardized
                    ORDER BY
                          is_defaulted_work_category ASC
                        , item_work_category_source
                        , specification_description
                        , effective_specification_description_standardized
                  ) AS rn
            FROM {DIM_ITEM_WORK_CATEGORY_TABLE}
        ) x
        WHERE rn = 1
    )

    , src AS (
        SELECT
              f.estimate_item_fact_key
            , f.project_key
            , f.item_specification_key

            , COALESCE(
                  f.project_classification_key
                , fp.project_classification_key
              ) AS project_classification_key

            , COALESCE(
                  f.county_key
                , fp.county_key
              ) AS county_key

            , f.project_id

            , dp.project_name
            , dp.project_number
            , dp.state_project_number
            , dp.control_section_job_csj
            , dp.controlling_project_id_ccsj
            , dp.project_type
            , dp.project_classification
            , dp.project_actual_let_date
            , dp.project_estimated_let_date
            , dp.project_limits_from
            , dp.project_limits_to
            , dp.county
            , dc.county_location AS location
            , dp.district_division
            , dp.highway
            , dp.short_description
            , dp.federal_project_number

            , f.bid_submit_sequence_number
            , f.bid_rank_sequence_number
            , f.low_bidder_flag
            , f.dbe_goal_percent
            , f.maximum_number_of_working

            , f.bid_code
            , f.alternative_bid_code
            , f.bid_item_sequence_number
            , f.bid_item_description
            , f.measurement_unit
            , f.bid_item_quantity
            , f.bid_item_unit_price_amount
            , f.bid_total_amount

            , f.specification_code
            , f.specification_description
            , ds.effective_specification_description
            , ds.effective_specification_description_standardized
            , f.spec_book_year

            , wc.item_work_category
            , wc.item_work_category_source
            , wc.is_defaulted_work_category

            , f.engineer_s_estimate_unit
            , f.sealed_engineer_s_estimate
            , f.sealed_engineer_s_estimate_1
            , f.a_b_engineer_s_estimate_amount
            , f.engineer_estimate_unit_price
            , f.engineer_project_total

            , f.force_account_amount
            , f.force_account_description
            , f.nbi_number
            , f.utility_id

            , f.business_submission_key
            , f.business_item_key

            , f.estimate_item_rank
            , f.estimate_item_version_count
            , f.estimate_item_has_multiple_versions
            , f.estimate_item_changed_across_versions

            , f.is_standard_specification_item
            , f.is_unmapped_specification_item
            , f.is_payment_adjustment_item
            , f.is_special_deduction_item
            , f.is_nonstandard_adjustment_item

            , CASE
                  WHEN f.bid_item_sequence_number IS NULL THEN TRUE
                  ELSE FALSE
              END AS is_nonstandard_estimate_item

            , CASE
                  WHEN f.bid_item_sequence_number IS NOT NULL THEN TRUE
                  ELSE FALSE
              END AS is_standard_comparison_item

            , f.extended_estimate_item_amount_calc
            , f.project_total_vs_extended_estimate_abs_diff

            , fp.max_engineer_project_total AS project_engineer_total
            , 'PROJECT_FACT_MAX' AS engineer_project_total_source
            , fp.min_engineer_project_total
            , fp.max_engineer_project_total
            , fp.estimate_item_extended_total
            , fp.estimate_total_vs_item_rollup_abs_diff
            , fp.has_conflicting_engineer_project_totals
            , fp.has_item_version_changes
            , fp.payment_adjustment_item_count AS project_payment_adjustment_item_count
            , fp.special_deduction_item_count AS project_special_deduction_item_count

            , CASE
                  WHEN fp.max_engineer_project_total IS NOT NULL
                   AND fp.max_engineer_project_total <> 0
                   AND f.extended_estimate_item_amount_calc IS NOT NULL
                  THEN f.extended_estimate_item_amount_calc / fp.max_engineer_project_total
                  ELSE NULL
              END AS estimate_item_amount_as_pct_of_project_total

            , CASE
                  WHEN fp.estimate_item_extended_total IS NOT NULL
                   AND fp.estimate_item_extended_total <> 0
                   AND f.extended_estimate_item_amount_calc IS NOT NULL
                  THEN f.extended_estimate_item_amount_calc / fp.estimate_item_extended_total
                  ELSE NULL
              END AS estimate_item_amount_as_pct_of_item_rollup_diagnostic

            , f.source_name
            , f.source_row_id
            , f.source_created_at
            , f.source_updated_at
            , f.source_version
            , f.ingestion_run_id
            , f.ingested_at_utc
            , f.record_hash

        FROM {FACT_ESTIMATE_ITEM_TABLE} f
        LEFT JOIN {DIM_PROJECT_TABLE} dp
            ON f.project_id = dp.project_id
        LEFT JOIN {DIM_COUNTY_TABLE} dc
            ON dp.county = dc.county
        LEFT JOIN {DIM_ITEM_SPEC_TABLE} ds
            ON f.item_specification_key = ds.item_specification_key
        LEFT JOIN work_category_map wc
            ON ds.effective_specification_description_standardized = wc.effective_specification_description_standardized
        LEFT JOIN {FACT_ESTIMATE_PROJECT_TABLE} fp
            ON f.project_id = fp.project_id
    )

    SELECT *
    FROM src
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_TABLE}
    """).collect()[0]["row_count"]

    missing_item_work_category = spark.sql(f"""
        SELECT COUNT(*) AS missing_count
        FROM {TARGET_TABLE}
        WHERE item_work_category IS NULL
    """).collect()[0]["missing_count"]

    missing_project_classification_keys = spark.sql(f"""
        SELECT COUNT(*) AS missing_count
        FROM {TARGET_TABLE}
        WHERE project_classification_key IS NULL
    """).collect()[0]["missing_count"]

    missing_county_keys = spark.sql(f"""
        SELECT COUNT(*) AS missing_count
        FROM {TARGET_TABLE}
        WHERE county_key IS NULL
    """).collect()[0]["missing_count"]

    log_run(
        "SUCCESS",
        row_count,
        "Built gold.fact_project_item_estimate successfully."
    )

    print("=" * 70)
    print("Built gold.fact_project_item_estimate")
    print(f"Row count: {row_count:,}")
    print("=" * 70)
    print("Validation:")
    print(f"  missing_item_work_category: {missing_item_work_category:,}")
    print(f"  missing_project_classification_keys: {missing_project_classification_keys:,}")
    print(f"  missing_county_keys: {missing_county_keys:,}")

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise